# 01 Preprocessing

This notebook performs the preprocessing pipeline for the dissertation project on **News Article Clustering by Topic Similarity**.

## Objectives
- Load the four cleaned base datasets
- Preprocess article text using **spaCy**
- Standardise the output into a common structure
- Combine all datasets into one dataframe
- Remove nulls, blanks, and exact duplicates
- Save the final dataset for vectorisation and clustering

## Datasets covered
- BBC News
- Reuters
- AG News
- Scraped News Articles


## 1. Import libraries

This section imports the required libraries for data loading, text preprocessing, and progress tracking.


In [ ]:
import pandas as pd
import spacy
from tqdm import tqdm

tqdm.pandas()

## 2. Load spaCy model

This project uses the `en_core_web_sm` model for tokenisation, stopword removal, and lemmatisation.

If the model is not installed in your environment, install it once before running the notebook:

```python
!python -m spacy download en_core_web_sm
```


In [ ]:
nlp = spacy.load("en_core_web_sm")

## 3. Define reusable preprocessing function

A single cleaning function is used across all datasets to keep preprocessing consistent.


In [ ]:
def clean_text_spacy(text):
    """Clean and lemmatise text using spaCy."""
    if pd.isna(text) or not isinstance(text, str):
        return ""

    doc = nlp(text.lower())
    tokens = [
        token.lemma_
        for token in doc
        if not token.is_stop and not token.is_punct and token.is_alpha
    ]
    return " ".join(tokens)

## 4. Define input files

Update these file names if your cleaned base files use different names or live in a different folder.


In [ ]:
bbc_file = "bbc_clean_base.xlsx"
reuters_file = "reuters_clean_base.xlsx"
agnews_file = "agnews_clean_base.xlsx"
scraped_file = "scraped_articles_clean_base.xlsx" 

## 5. Preprocess BBC dataset

In [ ]:
df_bbc = pd.read_excel(bbc_file)
print("BBC shape before preprocessing:", df_bbc.shape)
df_bbc.head()

In [ ]:
df_bbc["clean_text"] = df_bbc["text"].progress_apply(clean_text_spacy)
df_bbc["source"] = "bbc"

bbc_processed = df_bbc[["clean_text", "source"]].copy()
print("BBC shape after preprocessing:", bbc_processed.shape)
bbc_processed.head()

## 6. Preprocess Reuters dataset

In [ ]:
df_reuters = pd.read_excel(reuters_file)
print("Reuters shape before preprocessing:", df_reuters.shape)
df_reuters.head()

In [ ]:
df_reuters["clean_text"] = df_reuters["text"].progress_apply(clean_text_spacy)
df_reuters["source"] = "reuters"

reuters_processed = df_reuters[["clean_text", "source"]].copy()
print("Reuters shape after preprocessing:", reuters_processed.shape)
reuters_processed.head()

## 7. Preprocess AG News dataset

In [ ]:
df_ag = pd.read_excel(agnews_file)
print("AG News shape before preprocessing:", df_ag.shape)
df_ag.head()

In [ ]:
df_ag["clean_text"] = df_ag["description"].progress_apply(clean_text_spacy)
df_ag["source"] = "agnews"

ag_processed = df_ag[["clean_text", "source"]].copy()
print("AG News shape after preprocessing:", ag_processed.shape)
ag_processed.head()

## 8. Preprocess scraped news dataset

In [ ]:
df_scraped = pd.read_excel(scraped_file)
print("Scraped dataset shape before preprocessing:", df_scraped.shape)
df_scraped.head()

In [ ]:
df_scraped["clean_text"] = df_scraped["content"].progress_apply(clean_text_spacy)
df_scraped["source"] = "scraped"

scraped_processed = df_scraped[["clean_text", "source"]].copy()
print("Scraped dataset shape after preprocessing:", scraped_processed.shape)
scraped_processed.head()

## 9. Combine all preprocessed datasets

In [ ]:
combined_df = pd.concat(
    [bbc_processed, reuters_processed, ag_processed, scraped_processed],
    ignore_index=True
)

print("Combined shape before final cleaning:", combined_df.shape)
combined_df.head()

## 10. Remove nulls, blanks, and duplicates

In [ ]:
combined_df = combined_df[combined_df["clean_text"].notna()]
combined_df = combined_df[combined_df["clean_text"].str.strip() != ""]
combined_df = combined_df.drop_duplicates(subset="clean_text").reset_index(drop=True)

print("Combined shape after final cleaning:", combined_df.shape)
combined_df.head()

## 11. Save outputs

This saves:
- the combined preprocessed dataset
- the final cleaned and deduplicated dataset

If you only want one final file, you can keep the second export and remove the first.


In [ ]:
combined_preprocessed_file = "combined_preprocessed.xlsx"
final_clean_file = "combined_clean_deduplicated.xlsx"

combined_df.to_excel(combined_preprocessed_file, index=False)
combined_df.to_excel(final_clean_file, index=False)

print(f"Saved: {combined_preprocessed_file}")
print(f"Saved: {final_clean_file}")

## 12. Final summary

At this point, the dataset is ready for:
- TF-IDF vectorisation
- S-BERT embedding generation
- dimensionality reduction
- clustering experiments
